# 30 · A first look at the staged data

Notebooks 20–29 cleaned every source and saved one table per entity to `data/interim/staging/` (and to the `staging` schema once PostgreSQL is live). This notebook doesn't change anything. It's a guided tour so we know what we're working with before the analysis starts.

For each table: what one row means, how big it is, and a few real rows. A few sections end with a quick question answered straight from the data, to get a feel for the numbers. These are first looks, not findings. The careful versions (code bridges, NYC-only estimates, inflation adjustment) are built in the core layer.

Tables are grouped by the question they help answer:

1. Which jobs is NYC adding?
2. What do those jobs pay?
3. What education do they require?
4. How many graduates does each major produce?
5. The code crosswalks that tie majors and jobs together
6. Industries: employers, hiring, output
7. What NYC graduates actually do (Census microdata)
8. Skills, AI exposure and career clusters
9. Prices

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
STAGE = PROJECT / "data" / "interim" / "staging"
pd.set_option("display.max_columns", 30, "display.max_colwidth", 60, "display.width", 220,
              "display.float_format", "{:,.2f}".format)

log = pd.read_csv(STAGE / "_load_log.csv").drop_duplicates("table", keep="last").set_index("table")
_cache = {}


def load(table: str) -> pd.DataFrame:
    if table not in _cache:
        _cache[table] = pd.read_parquet(STAGE / f"{table}.parquet")
    return _cache[table]


def peek(table: str, cols: list[str] | None = None, n: int = 6, where=None, seed: int = 7):
    """Size, grain and a random sample of real rows (source_file hidden)."""
    df = load(table)
    meta = log.loc[table]
    display(Markdown(f"**`{table}`** · {len(df):,} rows × {df.shape[1]} columns · "
                     f"one row = `{meta['key']}` · source `{meta['source_key']}`"))
    view = df if where is None else df[where(df)]
    view = view[cols] if cols else view.drop(columns=["source_file"], errors="ignore")
    display(view.sample(min(n, len(view)), random_state=seed).sort_index())


print(f"{len(log)} staged tables, {int(log['rows'].sum()):,} rows in total")
display(log[["rows", "cols", "key", "source_key"]].sort_values("rows", ascending=False))

37 staged tables, 1,075,888 rows in total


,rows,cols,key,source_key
table,,,,
stg_census__acs_pums_person,279311,108,serialno + sporder,acs_pums
stg_bls__oews_research_ny,204386,19,year + naics + i_group + occ_code,bls_oews_research
stg_onet__descriptor_rating,123760,17,onet_soc_code + element_id + scale,onet
stg_bls__ind_occ_matrix,113898,15,industry_code + occ_code,bls_ind_occ_matrix
stg_ipeds__completions,58343,12,year + unitid + cip_code + major_number + award_level,ipeds_completions
stg_cuny__program_enrollment,53838,18,irp_code + college + award_name + period,cuny_programs
stg_cuny__program_graduates,53838,18,irp_code + college + award_name + period,cuny_programs
stg_census__qwi,39648,24,county_fips + naics_sector + breakdown + group_code + ye...,census_qwi
stg_onet__software,31821,9,onet_soc_code + product + commodity_id,onet


## 1. Which jobs is NYC adding?

**NYS DOL long-term projections**, four editions. Each edition gives a base-year and a ten-years-out employment estimate for every job, plus average annual openings. Openings matter more than growth: most openings come from people retiring or changing careers, not from new jobs. `quality_flag` marks the rows we know have problems (the 2020–30 edition started at the COVID low point, and 2022–32 published some jobs only as combined codes).

In [2]:
peek("stg_nysdol__occupation_projections",
     ["edition", "soc_code", "soc_title", "soc_level", "base_employment", "projected_employment",
      "pct_change", "annual_openings_avg", "quality_flag"],
     where=lambda d: d.soc_level == "detailed")

**`stg_nysdol__occupation_projections`** · 2,575 rows × 17 columns · one row = `edition + soc_code` · source `nysdol_projections`

,edition,soc_code,soc_title,soc_level,base_employment,projected_employment,pct_change,annual_openings_avg,quality_flag
150,2016-2026,21-1092,Probation Officers and Correctional Treatment Specialists,detailed,1080,1170,8.30,100,published
636,2016-2026,53-3099,"Motor Vehicle Operators, All Other",detailed,1220,1310,7.40,230,published
783,2018-2028,19-4091,"Environmental Science and Protection Technicians, Includ...",detailed,1330,1650,24.10,209,published
1285,2018-2028,53-7032,Excavating and Loading Machine and Dragline Operators,detailed,830,890,7.20,108,published
2019,2022-2032,17-3031,Surveying and Mapping Technicians,detailed,570,650,14.04,88,published
2526,2022-2032,51-9061,"Inspectors, Testers, Sorters, Samplers, and Weighers",detailed,2950,2730,-7.46,308,published


**Quick look:** the jobs with the most openings per year in NYC, 2022–32 edition.

In [3]:
p = load("stg_nysdol__occupation_projections")
(p[(p.edition == "2022-2032") & (p.soc_level == "detailed")]
 .nlargest(15, "annual_openings_avg")
 [["soc_code", "soc_title", "base_employment", "projected_employment", "pct_change", "annual_openings_avg"]]
 .reset_index(drop=True))

,soc_code,soc_title,base_employment,projected_employment,pct_change,annual_openings_avg
0,35-3023,Fast Food and Counter Workers,77730,91300,17.46,20647
1,37-2011,"Janitors and Cleaners, Except Maids and Housekeeping Cle...",93990,105450,12.19,15026
2,41-2031,Retail Salespersons,102310,95040,-7.11,13563
3,33-9032,Security Guards,90520,100130,10.62,13561
4,35-3031,Waiters and Waitresses,54830,62090,13.24,12816
5,41-2011,Cashiers,71340,64510,-9.57,12427
6,11-1021,General and Operations Managers,107930,117240,8.63,9781
7,43-6014,"Secretaries and Administrative Assistants, Except Legal,...",77620,84350,8.67,9653
8,43-4051,Customer Service Representatives,63730,63030,-1.10,8440
9,43-9061,"Office Clerks, General",68980,70020,1.51,8394


The same edition's Excel workbook adds columns the CSV doesn't have: 2026 wages and the state's "employment prospects" rating.

In [4]:
peek("stg_nysdol__projections_xlsx",
     ["soc_code", "soc_title", "wage_entry_2026", "wage_median_2026", "employment_prospects", "quality_flag"])

**`stg_nysdol__projections_xlsx`** · 657 rows × 13 columns · one row = `soc_code` · source `nysdol_projections_xlsx`

,soc_code,soc_title,wage_entry_2026,wage_median_2026,employment_prospects,quality_flag
193,25-2012,"Kindergarten Teachers, Except Special Education","56,947.72","79,221.24",Favorable,published
204,25-3000,Other Teachers and Instructors,NaN,NaN,NaN,published
272,29-1125,Recreational Therapists,"56,660.91","67,916.40",Very Favorable,published
285,29-1218,Obstetricians and Gynecologists,"114,055.81","340,480.48",Very Favorable,published
479,43-9021,Data Entry Keyers,"37,981.95","48,219.60",Very Unfavorable,published
597,51-7041,"Sawing Machine Setters, Operators, and Tenders, Wood","44,837.27","49,816.35",Very Unfavorable,published


## 2. What do those jobs pay?

**NYS DOL wages for NYC** (five boroughs, one current release). Gives entry, median and experienced pay. Entry pay is what a new graduate should expect.

In [5]:
peek("stg_nysdol__occupation_wages",
     ["soc_code", "soc_title", "soc_level", "employment", "entry_wage", "median_wage", "experienced_wage"],
     where=lambda d: d.soc_level == "detailed")

**`stg_nysdol__occupation_wages`** · 702 rows × 16 columns · one row = `area_code + soc_code` · source `nysdol_oews_regions`

,soc_code,soc_title,soc_level,employment,entry_wage,median_wage,experienced_wage
215,25-1126,"Philosophy and Religion Teachers, Postsecondary",detailed,710,68369,106342,150113
318,29-1243,Pediatric Surgeons,detailed,<NA>,246261,599425,<NA>
478,43-4071,File Clerks,detailed,1110,40225,47902,55257
486,43-4171,Receptionists and Information Clerks,detailed,27340,37654,46673,53309
584,49-9064,Watch Repairers,detailed,<NA>,89820,97747,97764
636,51-7041,"Sawing Machine Setters, Operators, and Tenders, Wood",detailed,<NA>,44837,49816,58094


**BLS OEWS, New York metro**, May 2018 to May 2025. This is our pay-over-time source. The area is the whole metro (NYC plus Long Island, the lower Hudson Valley and northern New Jersey), about twice the size of the city, so it's never presented as "NYC".

In [6]:
peek("stg_bls__oews_metro_occupations",
     ["year", "occ_code", "occ_title", "soc_edition", "employment", "annual_p10", "annual_p50", "annual_p90"],
     where=lambda d: d.occ_level == "detailed")

**`stg_bls__oews_metro_occupations`** · 6,281 rows × 28 columns · one row = `year + occ_code` · source `bls_oews_metro`

,year,occ_code,occ_title,soc_edition,employment,annual_p10,annual_p50,annual_p90
1829,2020,29-1123,Physical Therapists,hybrid,16250,"71,290.00","98,540.00","130,780.00"
2722,2021,35-9099,"Food Preparation and Serving Related Workers, All Other",2018,6570,"26,390.00","30,160.00","37,430.00"
3849,2022,53-2011,"Airline Pilots, Copilots, and Flight Engineers",2018,11380,"107,380.00","222,890.00",NaN
5243,2024,45-1011,"First-Line Supervisors of Farming, Fishing, and Forestry...",2018,430,"45,180.00","59,290.00","95,150.00"
5439,2024,51-9196,"Paper Goods Machine Setters, Operators, and Tenders",2018,2440,"37,050.00","47,250.00","63,190.00"
6162,2025,51-4034,"Lathe and Turning Machine Tool Setters, Operators, and T...",2018,270,"41,350.00","50,700.00","78,350.00"


**Quick look:** median pay for a few jobs CUNY graduates often go into, in nominal dollars (not yet inflation-adjusted). Software developers only appear from 2021. Before that, BLS reported them under a temporary combined code (`15-1256`), which is exactly the kind of gap the code bridges in the core layer fill in.

In [7]:
o = load("stg_bls__oews_metro_occupations")
picks = {"15-1252": "Software developers", "29-1141": "Registered nurses", "13-2011": "Accountants and auditors",
         "25-2021": "Elementary school teachers", "21-1021": "Child, family and school social workers",
         "29-2061": "Licensed practical nurses"}
(o[o.occ_code.isin(picks)]
 .assign(job=lambda d: d.occ_code.map(picks))
 .pivot_table(index="job", columns="year", values="annual_p50")
 .round(-2))

year,2018,2019,2020,2021,2022,2023,2024,2025
job,,,,,,,,
Accountants and auditors,"87,200.00","90,700.00","94,400.00","98,900.00","101,800.00","103,900.00","105,100.00","105,600.00"
"Child, family and school social workers","55,900.00","57,700.00","61,500.00","73,300.00","71,700.00","74,500.00","72,800.00","66,500.00"
Elementary school teachers,"80,100.00","80,100.00","81,200.00","79,300.00","83,300.00","97,200.00","95,200.00","96,900.00"
Licensed practical nurses,"56,200.00","57,000.00","57,800.00","60,100.00","62,700.00","66,300.00","72,800.00","75,900.00"
Registered nurses,"91,400.00","93,100.00","95,000.00","99,000.00","103,500.00","108,500.00","113,500.00","119,700.00"
Software developers,NaN,NaN,NaN,"129,000.00","136,600.00","149,100.00","162,000.00","166,800.00"


## 3. What education do these jobs require?

**BLS Employment Projections** (national, one edition). The typical education needed to *enter* each job, plus related experience and on-the-job training. This is what lets us say whether a job is realistic for someone with an associate's or a bachelor's degree.

In [8]:
peek("stg_bls__ep_entry_requirements", ["soc_code", "soc_title", "education", "experience", "training"])

**`stg_bls__ep_entry_requirements`** · 831 rows × 11 columns · one row = `soc_code` · source `bls_ep_education`

,soc_code,soc_title,education,experience,training
140,19-2021,Atmospheric and space scientists,Bachelor's degree,None,None
388,31-9095,Pharmacy aides,High school diploma or equivalent,None,Short-term on-the-job training
446,39-2011,Animal trainers,High school diploma or equivalent,None,Moderate-term on-the-job training
508,43-3099,"Financial clerks, all other",High school diploma or equivalent,None,Short-term on-the-job training
592,47-2181,Roofers,No formal educational credential,None,Moderate-term on-the-job training
682,51-2051,Fiberglass laminators and fabricators,High school diploma or equivalent,None,Moderate-term on-the-job training


In [9]:
e = load("stg_bls__ep_entry_requirements")
e.education.value_counts().rename("jobs").to_frame()

,jobs
education,
High school diploma or equivalent,326
Bachelor's degree,177
No formal educational credential,109
Doctoral or professional degree,73
Postsecondary nondegree award,51
Associate's degree,48
Master's degree,40
"Some college, no degree",7


What education people working in each job *actually* have (share of workers by attainment level).

In [10]:
peek("stg_bls__ep_attainment", ["soc_code", "soc_title", "attainment_level", "pct_of_workers"])

**`stg_bls__ep_attainment`** · 5,824 rows × 9 columns · one row = `soc_code + attainment_level` · source `bls_ep_education`

,soc_code,soc_title,attainment_level,pct_of_workers
913,19-1021,Biochemists and biophysicists,Associate's degree,0.00
2798,33-2021,Fire inspectors and investigators,Master's degree,5.60
3527,43-3021,Billing and posting clerks,Doctoral or professional degree,0.70
3622,43-4081,"Hotel, motel, and resort desk clerks",Associate's degree,11.30
3702,43-5021,Couriers and messengers,Doctoral or professional degree,1.00
4098,47-2131,"Insulation workers, floor, ceiling, and wall",Associate's degree,6.50


National growth and openings by job, which we use to fill gaps in the 2022–32 NYC edition (decision D1).

In [11]:
peek("stg_bls__ep_occupation_projections",
     ["soc_code", "soc_title", "employment_2025_k", "employment_2035_k", "employment_change_pct",
      "annual_openings_k", "median_wage_2025"])

**`stg_bls__ep_occupation_projections`** · 1,112 rows × 14 columns · one row = `soc_code` · source `bls_ep_occupation`

,soc_code,soc_title,employment_2025_k,employment_2035_k,employment_change_pct,annual_openings_k,median_wage_2025
46,11-9171,Funeral home managers,34.80,36.20,4.10,2.60,78790
53,13-1030,"Claims adjusters, appraisers, examiners, and investigators",389.70,367.90,-5.60,21.60,78020
179,19-1041,Epidemiologists,12.80,15.20,18.70,0.80,87220
318,25-2011,"Preschool teachers, except special education",583.20,609.40,4.50,66.70,38140
379,27-2031,Dancers,13.80,14.10,2.20,1.60,<NA>
403,27-4014,Sound engineering technicians,15.00,14.60,-3.10,1.00,73130


## 4. How many graduates does each major produce?

**IPEDS completions**: every degree awarded at every NYC college, by major (CIP code) and award level, award years 2015–2024. `major_number` 2 is a second major. The award-level codes we care about are 3 (associate's) and 5 (bachelor's).

In [12]:
peek("stg_ipeds__completions",
     ["year", "unitid", "cip_code", "cip_edition", "major_number", "award_level", "awards_total"])

**`stg_ipeds__completions`** · 58,343 rows × 12 columns · one row = `year + unitid + cip_code + major_number + award_level` · source `ipeds_completions`

,year,unitid,cip_code,cip_edition,major_number,award_level,awards_total
8184,2016,192439,13.1330,2010,1,5,0
25034,2019,190600,45.1101,2010,1,5,34
28771,2020,189097,05.0124,2020,2,5,1
33746,2020,195809,26.1201,2020,1,7,10
47240,2023,190424,26.0507,2020,1,7,1
47675,2023,190549,51.1508,2020,1,6,0


In [13]:
peek("stg_ipeds__institutions", ["year", "unitid", "name", "borough", "sector", "is_cuny"],
     where=lambda d: d.year == d.year.max())

**`stg_ipeds__institutions`** · 1,836 rows × 13 columns · one row = `year + unitid` · source `ipeds_directory`

,year,unitid,name,borough,sector,is_cuny
1695,2024,190549,CUNY Brooklyn College,Brooklyn,Public 4-year+,True
1749,2024,195173,St. Francis College,Brooklyn,Private nonprofit 4-year+,False
1761,2024,196884,Union Theological Seminary in the City of New York,Manhattan,Private nonprofit 4-year+,False
1797,2024,451413,Center for Allied Health Education,Brooklyn,Private for-profit 2-year,False
1818,2024,486026,Yeshiva Sholom Shachna,Brooklyn,Private nonprofit 4-year+,False
1826,2024,491774,Transitions Career Institute School of Nursing,Queens,Private for-profit < 2-year,False


**Quick look:** associate's and bachelor's degrees awarded by NYC colleges each year, CUNY and everyone else (first majors only).

In [14]:
c = load("stg_ipeds__completions")
inst = load("stg_ipeds__institutions")[["year", "unitid", "is_cuny"]]
(c[(c.major_number == 1) & (c.award_level.isin([3, 5]))]
 .merge(inst, on=["year", "unitid"], how="left")
 .assign(level=lambda d: d.award_level.map({3: "associate", 5: "bachelor"}),
         who=lambda d: d.is_cuny.map({True: "CUNY", False: "other NYC"}))
 .pivot_table(index="year", columns=["level", "who"], values="awards_total", aggfunc="sum"))

level associate           bachelor          
who        CUNY other NYC     CUNY other NYC
year                                        
2015      15478     10734    22736     29886
2016      16345     10107    23709     30475
2017      17372      9139    24456     29799
2018      18247      8791    25371     29785
2019      18880      8713    27273     30509
2020      18498      8319    27219     32114
2021      17596      8349    29590     31295
2022      14662      7554    28008     31749
2023      12246      6717    25074     30949
2024      11402      7088    23437     31298

**CUNY's own program data** goes back further (academic years 2007–08 to 2024–25) and names each program. Graduates and fall enrollment are two tables with the same program columns.

In [15]:
peek("stg_cuny__program_graduates",
     ["year", "college", "program_name", "award_name", "award_level", "cip_code", "graduates"],
     where=lambda d: d.graduates > 0)

**`stg_cuny__program_graduates`** · 53,838 rows × 18 columns · one row = `irp_code + college + award_name + period` · source `cuny_programs`

,year,college,program_name,award_name,award_level,cip_code,graduates
268,2024,Baruch College,Economics,BBA,bachelor,52.0601,68
2799,2017,Borough of Manhattan Community College,Psychology,AA,associate,42.0101,8
11623,2021,City College,Electronic Design and Multimedia,BFA,bachelor,09.0702,7
19600,2024,Hunter College,Urban Studies,BA,bachelor,45.1201,21
20658,2020,Hunter College,Childhood Special Education: Severe/Multiple Disabilities,MSED,graduate,13.1017,2
32929,2015,Lehman College,"Mathematics Teacher Education, Grades 7-12",BA,bachelor,13.1311,2


In [16]:
g = load("stg_cuny__program_graduates")
g.pivot_table(index="year", columns="award_level", values="graduates", aggfunc="sum").tail(10)

award_level,associate,bachelor,certificate,graduate
year,,,,
2016,16345,24504,159,9397
2017,17375,25270,117,9149
2018,18250,26164,301,9560
2019,18881,28117,337,10283
2020,18498,28023,286,10176
2021,17548,30244,288,11124
2022,14636,28557,284,10801
2023,12188,25788,266,10784
2024,11364,24085,317,10410


## 5. The crosswalks that tie it all together

Majors use CIP codes and jobs use SOC codes, and both changed versions during our time window. These tables are what connects them.

**CIP → SOC** (NCES): which jobs each major prepares students for. One major usually links to several jobs.

In [17]:
peek("stg_nces__cip_soc", ["cip_code", "cip_title", "soc_code", "soc_title"])

**`stg_nces__cip_soc`** · 6,097 rows × 6 columns · one row = `cip_code + soc_code` · source `nces_cip`

,cip_code,cip_title,soc_code,soc_title
1772,16.1501,Turkish Language and Literature.,25-1124,"Foreign Language and Literature Teachers, Postsecondary"
2639,30.2701,Human Biology.,25-1042,"Biological Science Teachers, Postsecondary"
3734,46.0412,Building/Construction Site Management/Manager.,49-9095,Manufactured Building and Mobile Home Installers
5086,52.1601,Taxation.,13-2022,Appraisers of Personal and Business Property
5279,60.0749,Urology Nurse Practitioner Residency/Fellowship Program.,29-1171,Nurse Practitioners
5979,99.9999,NO MATCH,43-5053,"Postal Service Mail Sorters, Processors, and Processing ..."


In [18]:
cs = load("stg_nces__cip_soc")
cs[cs.cip_code == "11.0701"][["cip_title", "soc_code", "soc_title"]]

,cip_title,soc_code,soc_title
656,Computer Science.,11-3021,Computer and Information Systems Managers
657,Computer Science.,15-1212,Information Security Analysts
658,Computer Science.,15-1221,Computer and Information Research Scientists
659,Computer Science.,15-1231,Computer Network Support Specialists
660,Computer Science.,15-1243,Database Architects
661,Computer Science.,15-1251,Computer Programmers
662,Computer Science.,15-1252,Software Developers
663,Computer Science.,15-1253,Software Quality Assurance Analysts and Testers
664,Computer Science.,15-1254,Web Developers
665,Computer Science.,15-1255,Web and Digital Interface Designers


Old-to-new code mappings: CIP 2010 → 2020, SOC 2010 → 2018, and the temporary hybrid codes BLS used in 2019–2020.

In [19]:
peek("stg_nces__cip_2010_2020", ["cip_2010", "cip_2010_title", "action", "cip_2020", "cip_2020_title"], n=4)
peek("stg_bls__soc_2010_2018", ["soc_2010", "soc_2010_title", "relation", "soc_2018", "soc_2018_title"], n=4)
peek("stg_bls__oews_hybrid", ["oes_2019", "oes_2019_title", "soc_2018", "soc_2018_title", "is_bls_aggregate"], n=4)

**`stg_nces__cip_2010_2020`** · 1,720 rows × 7 columns · one row = `cip_2010` · source `nces_cip`

,cip_2010,cip_2010_title,action,cip_2020,cip_2020_title
146,09.0402,Broadcast Journalism.,No substantive changes,09.0402,Broadcast Journalism.
458,16.0103,Language Interpretation and Translation.,No substantive changes,16.0103,Language Interpretation and Translation.
705,27.0101,"Mathematics, General.",No substantive changes,27.0101,"Mathematics, General."
1039,45.0901,International Relations and Affairs.,No substantive changes,45.0901,International Relations and Affairs.


**`stg_bls__soc_2010_2018`** · 900 rows × 6 columns · one row = `soc_2010 + soc_2018` · source `bls_soc_crosswalk`

,soc_2010,soc_2010_title,relation,soc_2018,soc_2018_title
256,25-2031,"Secondary School Teachers, Except Special and Career/Tec...",1:1,25-2031,"Secondary School Teachers, Except Special and Career/Tec..."
259,25-2052,"Special Education Teachers, Kindergarten and Elementary ...",split,25-2055,"Special Education Teachers, Kindergarten"
485,39-4011,Embalmers,1:1,39-4011,Embalmers
538,43-3099,"Financial Clerks, All Other",1:1,43-3099,"Financial Clerks, All Other"


**`stg_bls__oews_hybrid`** · 868 rows × 11 columns · one row = `oes_2019 + soc_2018 + oes_2018 + soc_2010` · source `bls_oews_hybrid`

,oes_2019,oes_2019_title,soc_2018,soc_2018_title,is_bls_aggregate
388,29-9092,Genetic Counselors,29-9092,Genetic Counselors,False
423,33-3051,Police and Sheriff's Patrol Officers,33-3051,Police and Sheriff’s Patrol Officers,False
518,43-1011,First-Line Supervisors of Office and Administrative Supp...,43-1011,First-Line Supervisors of Office and Administrative Supp...,False
785,51-9022,"Grinding and Polishing Workers, Hand",51-9022,"Grinding and Polishing Workers, Hand",False


The reference lists of every valid code: SOC 2018 (major → minor → broad → detailed) and CIP 2020.

In [20]:
peek("stg_bls__soc_2018_structure", n=4)
peek("stg_nces__cip_2020", ["cip_code", "level", "cip_title", "definition"], n=4)

**`stg_bls__soc_2018_structure`** · 1,447 rows × 7 columns · one row = `soc_code` · source `bls_soc_structure`

,soc_code,soc_title,level,major_code,minor_code,broad_code
562,29-1217,Neurologists,detailed,29-0000,29-1000,29-1210
571,29-1242,"Orthopedic Surgeons, Except Pediatric",detailed,29-0000,29-1000,29-1240
795,39-9032,Recreation Workers,detailed,39-0000,39-9000,39-9030
1309,51-9124,"Coating, Painting, and Spraying Machine Setters, Operato...",detailed,51-0000,51-9000,51-9120


**`stg_nces__cip_2020`** · 2,848 rows × 10 columns · one row = `cip_code` · source `nces_cip`

,cip_code,level,cip_title,definition
365,12.0599,detailed,"Culinary Arts and Related Services, Other.",Any instructional program in culinary arts and related s...
1235,30.5201,detailed,"Digital Humanities and Textual Studies, General.","A program that focuses on digital textual editing, archi..."
2041,51.2006,detailed,Clinical and Industrial Drug Development.,A program that focuses on the scientific application of ...
2781,61.1910,detailed,Pediatric Gastroenterology Fellowship Program.,A fellowship training program that prepares physicians i...


## 6. Industries: employers, hiring and output

**QCEW (NYS DOL)**: establishments, jobs and total wages by industry and borough, 2000–2025. The most complete count of jobs there is, because it comes from unemployment-insurance filings.

In [21]:
peek("stg_nysdol__qcew_annual",
     ["year", "area_name", "ownership", "naics_code", "naics_title", "establishments", "employment", "total_wages"],
     where=lambda d: d.naics_level == 2)

**`stg_nysdol__qcew_annual`** · 24,114 rows × 12 columns · one row = `area_code + ownership + naics_code + year` · source `nysdol_qcew_annual`

,year,area_name,ownership,naics_code,naics_title,establishments,employment,total_wages
5095,2023,Kings County,Private,61,Educational Services,1345,35761,1822272614
7462,2021,Kings County,"Total, All Ownerships",99,Unclassified,8339,5093,279318024
15132,2024,Queens County,"Total, All Ownerships",51,Information,765,7608,784520920
16463,2020,Richmond County,Private,44,Retail Trade,1277,14352,519607007
16631,2022,Richmond County,Private,48,Transportation and Warehousing,268,16952,1079535676
21987,2024,New York City,"Total, All Government",44,Retail Trade,5,61,2762153


In [22]:
peek("stg_bls__qcew_county", n=4)

**`stg_bls__qcew_county`** · 1,365 rows × 11 columns · one row = `county_fips + year + own_code + industry_code` · source `bls_qcew_county`

,county_fips,year,own_code,industry_code,agglvl_code,estabs,emp,total_wages,avg_annual_pay,disclosed
263,36047,2019,1,62,74,4,1347,124200089,92239,True
533,36047,2025,5,52,74,1709,13056,1574608982,120607,True
573,36061,2019,5,42,74,7499,73299,9200094546,125515,True
1335,36085,2025,2,10,71,40,2543,376515052,148074,True


**QWI (Census)**: hires, separations and new-hire pay by industry, borough and quarter, broken down by age or by education.

In [23]:
peek("stg_census__qwi",
     ["county_fips", "naics_sector", "breakdown", "group_code", "year", "quarter", "emp_begin",
      "hires_new", "separations", "earn_new_hire_stable_monthly"])

**`stg_census__qwi`** · 39,648 rows × 24 columns · one row = `county_fips + naics_sector + breakdown + group_code + year + quarter` · source `census_qwi`

,county_fips,naics_sector,breakdown,group_code,year,quarter,emp_begin,hires_new,separations,earn_new_hire_stable_monthly
1693,36005,23,age,A04,2022,2,2097,442,459,5191
17841,36061,31-33,age,A02,2020,2,270,77,102,1690
17879,36061,31-33,age,A03,2022,4,1015,267,227,4751
22563,36061,71,age,A02,2024,4,2224,764,722,1864
23094,36061,72,age,A07,2024,3,28599,2846,3570,3627
37911,36085,61,education,E5,2025,4,377,152,<NA>,<NA>


**CES (NYS DOL)**: monthly jobs by industry since 1990, the most up-to-date signal. BEA: GDP by industry for each borough.

In [24]:
peek("stg_nysdol__ces_monthly", ["series_title", "year", "month", "employment"], n=4)
peek("stg_bea__county_gdp", ["county_name", "line_title", "year", "gdp_current_thousands", "is_withheld"], n=4)

**`stg_nysdol__ces_monthly`** · 9,660 rows × 8 columns · one row = `series_code + year + month` · source `nysdol_ces`

,series_title,year,month,employment
5292,"Advertising, Public Relations, and Related Services",2023,1,71600
6576,Outpatient Care Centers,2022,9,23700
6787,Hospitals,2024,12,192600
9427,State Government Hospitals,2022,8,6500


**`stg_bea__county_gdp`** · 660 rows × 11 columns · one row = `county_fips + line_code + year` · source `bea_county_gdp`

,county_name,line_title,year,gdp_current_thousands,is_withheld
146,"Kings, NY","Agriculture, forestry, fishing and hunting",2021,2514,False
352,"New York, NY",Management of companies and enterprises,2023,19972383,False
647,"Richmond, NY",Accommodation and food services,2024,529201,False
658,"Richmond, NY",Government and government enterprises,2023,3594619,False


Which jobs make up each industry. The BLS research estimates give New York's own staffing pattern (for example, what share of hospital jobs are nurses). The national matrix adds 2035 projections.

In [25]:
peek("stg_bls__oews_research_ny",
     ["year", "naics", "naics_title", "occ_code", "occ_title", "tot_emp", "pct_total", "a_median"])
peek("stg_bls__ind_occ_matrix",
     ["industry_title", "occ_code", "occ_title", "emp_2025_k", "pct_of_industry_2025", "emp_2035_k"], n=4)

**`stg_bls__oews_research_ny`** · 204,386 rows × 19 columns · one row = `year + naics + i_group + occ_code` · source `bls_oews_research`

,year,naics,naics_title,occ_code,occ_title,tot_emp,pct_total,a_median
15905,2019,541330,Engineering Services,43-0000,Office and Administrative Support Occupations,4170,10.39,"52,490.00"
110697,2022,713200,Gambling Industries,33-0000,Protective Service Occupations,680,18.73,"38,570.00"
138037,2023,623000,Nursing and Residential Care Facilities,43-4111,"Interviewers, Except Eligibility and Loan",410,0.20,"53,690.00"
164402,2024,611000,Educational Services,31-1131,Nursing Assistants,950,0.09,"43,520.00"
180431,2025,339000,Miscellaneous Manufacturing,11-3051,Industrial Production Managers,230,0.86,"117,840.00"
191059,2025,541900,"Other Professional, Scientific, and Technical Services",15-0000,Computer and Mathematical Occupations,1210,2.66,"115,790.00"


**`stg_bls__ind_occ_matrix`** · 113,898 rows × 15 columns · one row = `industry_code + occ_code` · source `bls_ind_occ_matrix`

,industry_title,occ_code,occ_title,emp_2025_k,pct_of_industry_2025,emp_2035_k
18139,Pharmaceutical and medicine manufacturing,49-1000,"Supervisors of installation, maintenance, and repair wor...",1.60,0.50,1.70
34708,Retail trade,19-1013,Soil and plant scientists,0.20,0.00,0.20
46537,Freight transportation arrangement,19-5000,Occupational health and safety specialists and technicians,0.50,0.20,0.70
94168,"Arts, entertainment, and recreation",51-8021,Stationary engineers and boiler operators,0.20,0.00,0.20


## 7. What NYC graduates actually do (Census microdata)

**ACS PUMS**, 2020–2024 five-year file, New York City residents. One row is one survey respondent, and `weight` says how many New Yorkers that person stands for. The 80 `pwgtp*` columns are replicate weights for margins of error and are hidden here. Field of degree (`fod1`) is only asked of people with a bachelor's or higher.

In [26]:
peek("stg_census__acs_pums_person",
     ["borough", "age", "degree_level", "fod1", "socp", "naicsp", "employment_status",
      "wage_2024usd", "usual_hours", "weight"], n=8)

**`stg_census__acs_pums_person`** · 279,311 rows × 108 columns · one row = `serialno + sporder` · source `acs_pums`

,borough,age,degree_level,fod1,socp,naicsp,employment_status,wage_2024usd,usual_hours,weight
99350,Manhattan,70,bachelor,2300,131070,5411,6,0,<NA>,58
124280,Brooklyn,58,associate,NaN,NaN,NaN,6,0,<NA>,12
136545,Bronx,55,below associate,NaN,372012,7211,1,13406,40,9
138461,Brooklyn,50,bachelor,2400,333050,92MP,3,61446,40,47
154112,Queens,41,graduate,2304,291141,622M,1,67032,23,23
158297,Manhattan,59,bachelor,6402,1110XX,6243,3,279298,50,23
173541,Manhattan,72,bachelor,6203,113031,52M3,6,0,<NA>,46
198709,Queens,59,below associate,NaN,212021,8131,1,68216,40,19


**Quick look:** NYC residents aged 25 to 34, weighted, by highest degree.

In [27]:
a = load("stg_census__acs_pums_person")
young = a[a.age.between(25, 34)]
(young.groupby("degree_level").weight.sum()
 .rename("people (weighted)").to_frame()
 .assign(share=lambda d: d.iloc[:, 0] / d.iloc[:, 0].sum()))

,people (weighted),share
degree_level,,
associate,92453,0.06
bachelor,533112,0.37
below associate,526998,0.37
graduate,271445,0.19


## 8. Skills, AI exposure and career clusters

**O\*NET 31.0** describes every job in detail. One row is a job × descriptor × scale, for example "Programming, importance, 4.1 out of 5".

In [28]:
peek("stg_onet__occupation", ["onet_soc_code", "title", "description"], n=3)
peek("stg_onet__descriptor_rating", ["soc_code", "domain", "element_name", "scale", "value"])

**`stg_onet__occupation`** · 1,016 rows × 8 columns · one row = `onet_soc_code` · source `onet`

,onet_soc_code,title,description
24,11-3071.04,Supply Chain Managers,"Direct or coordinate production, purchasing, warehousing..."
682,43-6012.00,Legal Secretaries and Administrative Assistants,"Perform secretarial duties using legal terminology, proc..."
853,51-4041.00,Machinists,Set up and operate a variety of machine tools to produce...


**`stg_onet__descriptor_rating`** · 123,760 rows × 17 columns · one row = `onet_soc_code + element_id + scale` · source `onet`

,soc_code,domain,element_name,scale,value
42079,33-1012,skill_transferable,Equipment Maintenance,LV,0.75
50067,45-4011,skill_transferable,Technology Design,LV,0.38
52849,47-5013,skill_transferable,Management of Personnel Resources,LV,2.50
70084,13-2072,knowledge,Foreign Language,IM,1.64
106160,47-2011,knowledge,Building and Construction,IM,2.82
116914,51-6091,knowledge,Engineering and Technology,IM,2.41


In [29]:
peek("stg_onet__job_zone", ["onet_soc_code", "soc_code", "job_zone"], n=4)
peek("stg_onet__education_experience", ["soc_code", "question", "category_label", "share_pct"], n=4)
peek("stg_onet__software", ["soc_code", "product", "commodity_title", "is_hot_technology"], n=5)

**`stg_onet__job_zone`** · 923 rows × 7 columns · one row = `onet_soc_code` · source `onet`

,onet_soc_code,soc_code,job_zone
13,11-3031.00,11-3031,4
60,13-1031.00,13-1031,4
98,13-2071.00,13-2071,4
838,51-9011.00,51-9011,2


**`stg_onet__education_experience`** · 19,007 rows × 13 columns · one row = `onet_soc_code + scale + category` · source `onet`

,soc_code,question,category_label,share_pct
5663,41-4011,Required Level of Education,Post-Doctoral Training,0.00
15849,43-9021,Related Work Experience,Over 10 years,0.00
17369,51-2051,Related Work Experience,Up to and including 1 month,0.00
18984,53-7073,Related Work Experience,Over 10 years,0.00


**`stg_onet__software`** · 31,821 rows × 9 columns · one row = `onet_soc_code + product + commodity_id` · source `onet`

,soc_code,product,commodity_title,is_hot_technology
2475,11-9111,SmugMug Flickr,Graphics or photo imaging software,False
4213,13-1131,Microsoft PowerPoint,Presentation software,True
15432,17-3031,Database software,Data base user interface and query software,False
16150,19-1042,Microsoft Visual Basic,Development environment software,True
20399,25-4011,Microsoft Office software,Office suite software,True


**AI exposure** (Eloundou et al., 2024). Each task in each job was rated for how much a GPT-4-class model could speed it up. `beta_gpt4` rolls that up to the job, from 0 (no exposure) to 1.

In [30]:
peek("stg_eloundou__task_exposure", ["soc_code", "task", "gpt4_label"], n=4)
peek("stg_eloundou__job_exposure", ["soc_code", "title", "beta_gpt4", "beta_human"])

**`stg_eloundou__task_exposure`** · 19,265 rows × 8 columns · one row = `onet_soc_code + task_id` · source `eloundou_ai_exposure`

,soc_code,task,gpt4_label
595,11-9013,"Direct and monitor trapping and spawning of fish, egg in...",E0
12562,41-9041,Answer telephone calls from potential customers who have...,E1
14703,47-4061,Operate track wrenches to tighten or loosen bolts at joi...,E0
17622,51-9021,Turn valves to regulate the moisture contents of materials.,E0


**`stg_eloundou__job_exposure`** · 923 rows × 11 columns · one row = `onet_soc_code` · source `eloundou_ai_exposure`

,soc_code,title,beta_gpt4,beta_human
13,11-3031,Financial Managers,0.48,0.43
60,13-1031,"Claims Adjusters, Examiners, and Investigators",0.52,0.49
98,13-2071,Credit Counselors,0.62,0.69
209,19-1032,Foresters,0.37,0.37
825,51-7041,"Sawing Machine Setters, Operators, and Tenders, Wood",0.06,0.01
838,51-9011,Chemical Equipment Operators and Tenders,0.15,0.07


In [31]:
j = load("stg_eloundou__job_exposure")
pd.concat([j.nlargest(8, "beta_gpt4").assign(end="most exposed"),
           j.nsmallest(8, "beta_gpt4").assign(end="least exposed")])[["end", "soc_code", "title", "beta_gpt4"]]

,end,soc_code,title,beta_gpt4
132,most exposed,15-2021,Mathematicians,1.00
639,most exposed,43-9081,Proofreaders and Copy Markers,0.97
128,most exposed,15-1299,Blockchain Engineers,0.97
601,most exposed,43-4021,Correspondence Clerks,0.96
371,most exposed,27-3092,Court Reporters and Simultaneous Captioners,0.96
116,most exposed,15-1251,Computer Programmers,0.95
114,most exposed,15-1243,Data Warehousing Specialists,0.94
112,most exposed,15-1242,Database Administrators,0.94
355,least exposed,27-2021,Athletes and Sports Competitors,0.00
471,least exposed,31-1132,Orderlies,0.00


**Career Clusters** (Advance CTE) group majors, jobs and industries into 14 broad career areas and 72 sub-clusters, so the dashboard can say "Health Science" instead of listing 40 job codes.

In [32]:
peek("stg_cte__cluster", n=5)
peek("stg_cte__soc_subcluster", ["soc_code", "soc_title", "cluster_name", "subcluster_name"], n=4)
peek("stg_cte__cip_subcluster", ["cip_code", "cip_title", "cluster_name", "subcluster_name"], n=4)
peek("stg_cte__naics_subcluster", ["naics_sector", "naics_title", "cluster_name", "subcluster_name"], n=4)

**`stg_cte__cluster`** · 72 rows × 3 columns · one row = `subcluster_name` · source `career_clusters`

,cluster_name,subcluster_name
2,Advanced Manufacturing,Production & Automation
20,Construction,Equipment Operation & Maintenance
21,Construction,Skilled Trades
61,Public Service & Safety,Emergency Response
62,Public Service & Safety,Judicial Systems


**`stg_cte__soc_subcluster`** · 1,203 rows × 5 columns · one row = `soc_code + subcluster_name` · source `career_clusters`

,soc_code,soc_title,cluster_name,subcluster_name
141,15-1212,Information Security Analysts,Digital Technology,Network Systems & Cybersecurity
367,19-3093,Historians,Education,"Teaching, Training, & Facilitation"
855,45-2099,"Agricultural Workers, All Other",Agriculture,Agricultural Technology & Automation
1071,51-6093,Upholsterers,Advanced Manufacturing,Production & Automation


**`stg_cte__cip_subcluster`** · 4,725 rows × 5 columns · one row = `cip_code + subcluster_name` · source `career_clusters`

,cip_code,cip_title,cluster_name,subcluster_name
661,13.0603,Educational Statistics and Research Methods.,Management & Entrepreneurship,Business Information Management
2633,43.0203,Fire Science/Fire-fighting.,Public Service & Safety,Emergency Response
3255,51.0509,Pediatric Dentistry/Pedodontics.,Education,"Teaching, Training, & Facilitation"
4164,61.0105,Family Medicine/Osteopathic Neuromusculoskeletal Medicin...,Education,"Teaching, Training, & Facilitation"


**`stg_cte__naics_subcluster`** · 112 rows × 5 columns · one row = `naics_sector + subcluster_name` · source `career_clusters`

,naics_sector,naics_title,cluster_name,subcluster_name
78,44-45,Retail Trade,"Hospitality, Events, & Tourism",Travel & Leisure
82,55,Management of Companies and Enterprises,Management & Entrepreneurship,Entrepreneurship & Small Business
87,54,"Professional, Scientific, and Technical Services",Marketing & Sales,Retail & Customer Experience
96,56,Administrative and Support and Waste Management and Reme...,Public Service & Safety,Public Safety


## 9. Prices

**CPI-U, New York metro** (with the U.S. average for comparison). Period `M13` is the annual average, which is what we use to convert wages to 2025 dollars. October 2025 is missing because BLS collected no data during the federal shutdown.

In [33]:
cpi = load("stg_bls__cpi")
(cpi[cpi.period == "M13"]
 .pivot_table(index="year", columns="area_name", values="index_value"))

area_name,"New York-Newark-Jersey City, NY-NJ-PA",U.S. city average
year,,
2015,260.56,237.02
2016,263.37,240.01
2017,268.52,245.12
2018,273.64,251.11
2019,278.16,255.66
2020,282.92,258.81
2021,292.30,270.97
2022,310.14,292.65
2023,322.00,304.70


## What's next

This is the raw material. The core layer (dbt) turns it into one model: a job table every source attaches to, a major table, the major → job and industry → job bridges, wages in 2025 dollars, and NYC-only estimates where only metro data exists. Then the dashboard reads from that.